# Ch 12. トークナイザー — 解答

> このノートブックは5問すべての厳密な解答と再現可能な検証を含む。

## 問題 1

**問題**: BPE , .

### 厳密な解説

制御する要因: **BPE merge count on a fixed Korean corpus**。  測定指標: **token count and learned merge sequence**。  解釈と限界: BPE repeatedly merges the most frequent adjacent symbol pair. Korean syllable/morpheme variation fragments counts; ties also make results corpus/order dependent.


In [ ]:
import torch
from collections import Counter
words=[list(w) for w in '낮 말은 새가 듣고 밤 말은 쥐가 듣는다'.split()]
for step in range(5):
 pairs=Counter((a,b) for w in words for a,b in zip(w,w[1:])); pair=max(pairs,key=lambda p:(pairs[p],p)); merged=''.join(pair); words=[[merged if i<len(w)-1 and (w[i],w[i+1])==pair else w[i] for i in range(len(w)) if i==0 or (w[i-1],w[i])!=pair] for w in words]; print({'step':step+1,'merge':pair,'tokens':sum(map(len,words))})


## 問題 2

**問題**: BPE WordPiece 学習 比較.

### 厳密な解説

制御する要因: **selection score: BPE pair frequency versus WordPiece association score**。  測定指標: **first selected pair**。  解釈と限界: BPE favors raw frequent pairs; WordPiece-style scoring divides by unigram frequencies and can favor a more exclusive pair. Same corpus makes the objective the only changed factor.


In [ ]:
import torch
from collections import Counter
seq=[tuple(w) for w in 'low lower newest widest low low'.split()]; pairs=Counter((a,b) for w in seq for a,b in zip(w,w[1:])); uni=Counter(c for w in seq for c in w); bpe=max(pairs,key=lambda p:pairs[p]); wp=max(pairs,key=lambda p:pairs[p]/(uni[p[0]]*uni[p[1]])); print({'BPE_first':bpe,'WordPiece_first':wp,'BPE_count':pairs[bpe],'WordPiece_score':pairs[wp]/(uni[wp[0]]*uni[wp[1]])})


## 問題 3

**問題**: 100, 500, 1000 比較.

### 厳密な解説

制御する要因: **BPE vocabulary budget: 100, 500, 1000 (capped by available pairs)**。  測定指標: **mean tokens per sentence**。  解釈と限界: The same local corpus is tokenized after increasing merge budgets. A tiny corpus exhausts useful pairs before large nominal vocabulary sizes, which is an explicit limitation.


In [ ]:
import torch
from collections import Counter
base=['the quick brown fox','the quicker brown fox','quick foxes jump']*20
for budget in (100,500,1000):
 words=[list(w) for s in base for w in s.split()]; merges=min(budget-30,40)
 for _ in range(merges):
  pairs=Counter((a,b) for w in words for a,b in zip(w,w[1:]));
  if not pairs: break
  p=max(pairs,key=pairs.get); m=''.join(p); words=[[m if i<len(w)-1 and (w[i],w[i+1])==p else w[i] for i in range(len(w)) if i==0 or (w[i-1],w[i])!=p] for w in words]
 print({'vocab_budget':budget,'mean_tokens_per_sentence':sum(map(len,words))/len(base)})


## 問題 4

**問題**: Byte-level BPE UNK .

### 厳密な解説

制御する要因: **input Unicode strings**。  測定指標: **UTF-8 byte round-trip failures and byte vocabulary bound**。  解釈と限界: Every Unicode string maps to bytes in 0..255, so a byte-level base vocabulary has no UNK. Merges only combine known byte sequences and cannot remove coverage.


In [ ]:
import torch
samples=['한글','🙂','𝛑','mixed언어']; rows=[]
for s in samples: ids=list(s.encode('utf-8')); rows.append((s,ids,bytes(ids).decode('utf-8')==s))
print({'base_vocabulary_size':256,'round_trips':rows}); assert all(ok for _,_,ok in rows)


## 問題 5

**問題**: HuggingFace `tokenizers` GPT-2 トークナイザー エンコード.

### 厳密な解説

制御する要因: **GPT-2 byte-level pretokenization of Korean text**。  測定指標: **UTF-8 byte IDs and reversible decoded text**。  解釈と限界: A pretrained GPT-2 merge table is not bundled and downloads are forbidden, so the cell executes the tokenizer's lossless byte-level base stage rather than pretending pretrained token IDs were loaded. Supplying local vocab.json/merges.txt is the explicit boundary for exact GPT-2 IDs.


In [ ]:
import torch
text='안녕하세요, GPT-2!'; byte_ids=list(text.encode('utf-8')); decoded=bytes(byte_ids).decode('utf-8'); print({'text':text,'utf8_byte_ids':byte_ids,'decoded':decoded,'exact_gpt2_ids':'requires local vocab.json and merges.txt'}); assert decoded==text
